# 01 · Decorators

**Module 09 – Advanced Functions** · Estimated time: 60 min

> Merges 📗 *Decorators* (builds the concept step by step) with 📘 *Decorators in Python* (arguments, chaining, use cases). ⚠️ Adds what both are missing: `*args/**kwargs` in wrappers, **`functools.wraps`**, returning values, class-based decorators, and built-in decorators.

### 🎯 What you will learn
1. The building blocks: functions are objects, nested functions, returning functions, functions as arguments
2. Writing a decorator by hand, then with `@`
3. ✅ The correct general template (`*args`, `**kwargs`, `return`, `@functools.wraps`)
4. Decorators that take arguments
5. Stacking (chaining) decorators: order matters
6. Real-world decorators: timing, logging, caching, retry, validation, registration
7. Class-based decorators and decorating classes
8. Built-in decorators you already know

## 1. Building blocks (📗)

### Functions are objects

In [1]:
def hello(name="Jose"):
    return "Hello " + name

greet = hello            # assign the function itself (no parentheses)
print(greet())
del hello                # delete the original NAME...
print(greet())           # ...the function object still exists via 'greet'

Hello Jose
Hello Jose


### Functions inside functions

In [2]:
def hello(name="Jose"):
    print("The hello() function has been executed")

    def greet():
        return "\t This is inside the greet() function"

    def welcome():
        return "\t This is inside the welcome() function"

    print(greet())
    print(welcome())
    print("Now we are back inside the hello() function")

hello()

The hello() function has been executed
	 This is inside the greet() function
	 This is inside the welcome() function
Now we are back inside the hello() function


In [3]:
welcome()     # NameError: welcome only exists inside hello()

NameError: name 'welcome' is not defined

### Returning functions

In [4]:
def hello(name="Jose"):
    def greet():
        return "\t This is inside the greet() function"
    def welcome():
        return "\t This is inside the welcome() function"
    return greet if name == "Jose" else welcome

x = hello()
print(x)          # a function object
print(x())        # call it

<function hello.<locals>.greet at 0x11028c0f0>
	 This is inside the greet() function


### Functions as arguments

In [5]:
def hi():
    return "Hi Jose!"

def other(func):
    print("Other code would go here")
    print(func())

other(hi)

Other code would go here
Hi Jose!


## 2. Creating a decorator (📗)

A **decorator** is a function that **takes a function and returns a new function** that adds behaviour *around* the original, without modifying its code.

In [6]:
def new_decorator(func):
    def wrap_func():
        print("Code would be here, before executing the func")
        func()
        print("Code here will execute after the func()")
    return wrap_func

def func_needs_decorator():
    print("This function is in need of a Decorator")

func_needs_decorator()
print("---")
func_needs_decorator = new_decorator(func_needs_decorator)   # manual decoration
func_needs_decorator()

This function is in need of a Decorator
---
Code would be here, before executing the func
This function is in need of a Decorator
Code here will execute after the func()


The `@` syntax is just **syntactic sugar** for `f = decorator(f)`:

In [7]:
@new_decorator
def func_needs_decorator():
    print("This function is in need of a Decorator")

func_needs_decorator()

Code would be here, before executing the func
This function is in need of a Decorator
Code here will execute after the func()


## 3. ✅ The correct general-purpose template

The simple wrapper above has three problems:
1. It only works for functions with **no arguments**
2. It throws away the **return value**
3. It hides the original function's **name and docstring**

In [8]:
def naive(func):
    def wrapper():
        func()
    return wrapper

@naive
def add(a, b):
    """Add two numbers."""
    return a + b

try:
    add(2, 3)
except TypeError as e:
    print("TypeError:", e)
print(add.__name__, add.__doc__)     # 'wrapper', None: metadata lost

TypeError: naive.<locals>.wrapper() takes 0 positional arguments but 2 were given
wrapper None


✅ **Use this template for every decorator:**

In [9]:
import functools

def my_decorator(func):
    @functools.wraps(func)                 # copy __name__, __doc__, etc. from func
    def wrapper(*args, **kwargs):          # accept ANY arguments
        # ... before ...
        result = func(*args, **kwargs)     # call the original, passing everything through
        # ... after ...
        return result                      # return its result
    return wrapper

@my_decorator
def add(a, b):
    """Add two numbers."""
    return a + b

print(add(2, 3), add.__name__, add.__doc__)

5 add Add two numbers.


## 4. Practical decorators

### Timing

In [10]:
import time

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.perf_counter() - start:.4f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(i * i for i in range(n))

print(slow_sum(1_000_000))

slow_sum took 0.0269s
333332833333500000


### Logging calls

In [11]:
def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        arg_text = ", ".join([repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()])
        print(f"→ {func.__name__}({arg_text})")
        result = func(*args, **kwargs)
        print(f"← {func.__name__} returned {result!r}")
        return result
    return wrapper

@log_calls
def power(base, exp=2):
    return base ** exp

power(3)
power(2, exp=10)

→ power(3)
← power returned 9
→ power(2, exp=10)
← power returned 1024


1024

### Access control / validation

In [12]:
def require_admin(func):
    @functools.wraps(func)
    def wrapper(user, *args, **kwargs):
        if user.get("role") != "admin":
            raise PermissionError(f"{user.get('name')} is not an admin")
        return func(user, *args, **kwargs)
    return wrapper

@require_admin
def delete_database(user):
    return "Database deleted!"

print(delete_database({"name": "Asha", "role": "admin"}))
try:
    delete_database({"name": "Ravi", "role": "viewer"})
except PermissionError as e:
    print("PermissionError:", e)

Database deleted!
PermissionError: Ravi is not an admin


### Caching (memoization), built in!

In [13]:
@functools.cache                         # or @functools.lru_cache(maxsize=128)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(80), fib.cache_info())

23416728348467685 CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)


## 5. Decorators with arguments (📘)

Needs **one more level**: a function that takes the arguments and *returns a decorator*.

```
@repeat(3)          →   greet = repeat(3)(greet)
   └─ repeat(3) returns the real decorator, which then wraps greet
```

In [14]:
def repeat(n):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def greet(name):
    print(f"Hello, {name}!")

greet("world")

Hello, world!
Hello, world!
Hello, world!


> ⚠️ **Fix to the handbook example:** its wrapper is `def wrapper(a): ... func(a)`, which only works for functions with exactly **one** argument (despite the text saying "any number of arguments"). The version above uses `*args, **kwargs`.

### A retry decorator (very common in real code)

In [15]:
import random

def retry(times=3, exceptions=(Exception,), delay=0.0):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"  attempt {attempt} failed: {e}")
                    if attempt == times:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

random.seed(11)

@retry(times=5, exceptions=(ConnectionError,))
def flaky_request():
    if random.random() < 0.7:
        raise ConnectionError("network glitch")
    return "200 OK"

print(flaky_request())

  attempt 1 failed: network glitch
  attempt 2 failed: network glitch
200 OK


## 6. Chaining decorators (📘)

Decorators apply **bottom-up** (closest to the function first) and execute **top-down** when called:

In [16]:
def uppercase(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper

def exclaim(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) + "!!!"
    return wrapper

@uppercase
@exclaim
def greet():
    return "hello"

print(greet())                 # same as uppercase(exclaim(greet))

HELLO!!!


## 7. Registration decorators (how Flask/FastAPI routes work)

A decorator doesn't have to wrap anything; it can just **record** the function:

In [17]:
ROUTES = {}

def route(path):
    def decorator(func):
        ROUTES[path] = func
        return func                 # return the original unchanged
    return decorator

@route("/")
def home():
    return "Welcome home"

@route("/about")
def about():
    return "About us"

for path, handler in ROUTES.items():
    print(path, "→", handler())

/ → Welcome home
/about → About us


This is exactly the pattern behind web frameworks:
```python
@app.get("/users/{id}")        # FastAPI
def get_user(id: int): ...
```

## 8. Class-based decorators & decorating classes

In [18]:
class CountCalls:
    """A decorator implemented as a class: keeps state across calls."""
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.calls = 0
    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)

@CountCalls
def say(msg):
    return msg

say("a"); say("b"); say("c")
print(say.calls)

3


In [19]:
def add_repr(cls):
    """A class decorator: adds a __repr__ automatically."""
    def __repr__(self):
        fields = ", ".join(f"{k}={v!r}" for k, v in vars(self).items())
        return f"{cls.__name__}({fields})"
    cls.__repr__ = __repr__
    return cls

@add_repr
class User:
    def __init__(self, name, age):
        self.name, self.age = name, age

print(User("Asha", 30))

User(name='Asha', age=30)


## 9. Built-in decorators you already use

| Decorator | Purpose | Lesson |
|---|---|---|
| `@property`, `@x.setter` | managed attributes | Module 06 |
| `@classmethod`, `@staticmethod` | method types | Module 06 |
| `@abstractmethod` | abstract methods | Module 06 |
| `@dataclass` | generate boilerplate | Module 06 |
| `@functools.cache`, `@lru_cache` | memoization | here |
| `@functools.wraps` | preserve metadata | here |
| `@functools.total_ordering` | fill comparisons | Module 06 |
| `@contextlib.contextmanager` | generator → context manager | Module 09 · 03 |
| `@warnings.deprecated` 🆕 3.13 | mark deprecated APIs | Module 07 |
| `@typing.override` 🆕 3.12 | mark overriding methods for type checkers | |

## Recap: common uses (📘)
Logging · Timing · Authentication/authorisation · Caching · Rate limiting · Input validation · Retries · Registration/routing · Instrumentation. Frameworks like **Flask**, **Django** and **FastAPI** use decorators extensively.

## 📝 Summary
- A decorator takes a function and returns a (usually wrapped) function; `@d` means `f = d(f)`.
- ✅ Template: `@functools.wraps(func)` + `wrapper(*args, **kwargs)` + `return func(*args, **kwargs)`.
- Decorators with arguments need a third level (a decorator factory).
- Stacked decorators apply bottom-up.

## 🏋️ Practice (📗 Decorators homework is open-ended: explore Flask/FastAPI routes!)
1. Write `@debug` that prints the arguments and return value of any function.
2. Write `@validate_positive` that raises `ValueError` if any numeric argument is negative.
3. Write `@rate_limit(calls=2)` that allows a function to be called at most `calls` times, then raises `RuntimeError`.
4. Write `@timer(unit="ms")` that reports milliseconds or seconds.

<details><summary>Show solutions</summary>

```python
import functools, time

def debug(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"{func.__name__}{args}{kwargs} -> {result!r}")
        return result
    return wrapper

def validate_positive(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for v in [*args, *kwargs.values()]:
            if isinstance(v, (int, float)) and v < 0:
                raise ValueError(f"negative argument: {v}")
        return func(*args, **kwargs)
    return wrapper

def rate_limit(calls):
    def decorator(func):
        count = 0
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            nonlocal count
            if count >= calls:
                raise RuntimeError("rate limit exceeded")
            count += 1
            return func(*args, **kwargs)
        return wrapper
    return decorator

def timer(unit="s"):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            t = time.perf_counter()
            r = func(*args, **kwargs)
            dt = time.perf_counter() - t
            print(f"{func.__name__}: {dt * 1000:.2f} ms" if unit == "ms" else f"{dt:.4f} s")
            return r
        return wrapper
    return decorator
```
</details>